In [3]:
%pip install pandas numpy scikit-learn imbalanced-learn matplotlib seaborn joblib openpyxl

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import glob
import warnings
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [5]:
DATASET_DIR = "datasets"

files = glob.glob(os.path.join(DATASET_DIR, "*"))

print("Files found:")
for f in files:
    print(" -", f)

Files found:


In [6]:
datasets = {}

for file in files:
    try:
        filename = os.path.basename(file)
        
        if file.lower().endswith(".csv"):
            df = pd.read_csv(file)
        
        elif file.lower().endswith((".xlsx", ".xls")):
            df = pd.read_excel(file)
        
        else:
            continue
        
        datasets[filename] = df
        
        print(f"\nLoaded: {filename}")
        print("Shape:", df.shape)
        
    except Exception as e:
        print(f"ERROR loading {file}: {e}")

In [7]:
for name, df in datasets.items():
    print("\n" + "=" * 80)
    print("DATASET:", name)
    print("=" * 80)
    
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    
    print("\nColumns:")
    print(df.columns.tolist())
    
    print("\nData types:")
    print(df.dtypes)
    
    print("\nMissing values:")
    print(df.isnull().sum().sort_values(ascending=False).head(20))
    
    print("\nDuplicate rows:", df.duplicated().sum())
    
    print("\nFirst 3 rows:")
    display(df.head(3))

In [8]:
possible_targets = [
    "class",
    "Class",
    "fraud",
    "Fraud",
    "isFraud",
    "is_fraud",
    "fraud_flag",
    "label",
    "Label",
    "target",
    "Target"
]

for name, df in datasets.items():
    matches = [col for col in df.columns if col in possible_targets]
    
    print(f"{name}")
    print("Possible target columns:", matches)

In [9]:
for name, df in datasets.items():
    
    target_candidates = [
        col for col in df.columns
        if col in possible_targets
    ]
    
    if target_candidates:
        target = target_candidates[0]
        
        print("\n" + "=" * 60)
        print(name)
        print("Target:", target)
        
        print(df[target].value_counts())
        print("\nPercentage:")
        print(df[target].value_counts(normalize=True) * 100)

In [30]:
MAIN_DATASET = "dataset1.csv"

df = datasets[MAIN_DATASET].copy()

print("Selected dataset:", MAIN_DATASET)
print("Shape:", df.shape)

display(df.head())

KeyError: 'dataset1.csv'

In [11]:
TARGET_COLUMN = "Class"

print(df[TARGET_COLUMN].value_counts())

NameError: name 'df' is not defined

In [12]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

NameError: name 'df' is not defined

In [13]:
empty_columns = df.columns[df.isnull().all()].tolist()

print("Completely empty columns:", empty_columns)

df = df.drop(columns=empty_columns)

print("New shape:", df.shape)

NameError: name 'df' is not defined

In [14]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

NameError: name 'df' is not defined

In [15]:
id_like_columns = []

for col in X.columns:
    unique_ratio = X[col].nunique() / len(X)
    
    if (
        "id" in col.lower()
        and unique_ratio > 0.90
    ):
        id_like_columns.append(col)

print("Potential ID columns:")
print(id_like_columns)

NameError: name 'X' is not defined

In [16]:
X = X.drop(columns=id_like_columns, errors="ignore")

print("Features remaining:", X.shape[1])

NameError: name 'X' is not defined

In [17]:
numeric_features = X.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumerical:")
print(numeric_features)

print("\nCategorical:")
print(categorical_features)

NameError: name 'X' is not defined

In [18]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created.")

NameError: name 'numeric_features' is not defined

In [19]:
logistic_model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

print("Training Logistic Regression...")

logistic_model.fit(X_train, y_train)

print("Training completed.")

NameError: name 'preprocessor' is not defined

In [20]:
decision_tree_model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", DecisionTreeClassifier(
        max_depth=15,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

print("Training Decision Tree...")

decision_tree_model.fit(X_train, y_train)

print("Training completed.")

NameError: name 'preprocessor' is not defined

In [21]:
random_forest_model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

print("Training Random Forest...")

random_forest_model.fit(X_train, y_train)

print("Training completed.")

NameError: name 'preprocessor' is not defined

In [22]:
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree_model,
    "Random Forest": random_forest_model
}

results = []

for name, model in models.items():
    
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    
    y_pred = model.predict(X_test)
    
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = None
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test, y_pred, zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred, zero_division=0
    )
    
    if y_prob is not None:
        roc_auc = roc_auc_score(y_test, y_prob)
        pr_auc = average_precision_score(y_test, y_prob)
    else:
        roc_auc = np.nan
        pr_auc = np.nan
    
    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    })
    
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

NameError: name 'logistic_model' is not defined

In [23]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

display(results_df)

NameError: name 'results' is not defined

In [24]:
best_model_name = results_df.iloc[0]["Model"]

best_model = models[best_model_name]

y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

NameError: name 'results_df' is not defined

In [25]:
MODEL_PATH = "fraud_detection_model.pkl"

joblib.dump(
    best_model,
    MODEL_PATH
)

print("Best model:", best_model_name)
print("Saved to:", MODEL_PATH)

NameError: name 'best_model' is not defined

In [26]:
loaded_model = joblib.load(
    "fraud_detection_model.pkl"
)

print("Model loaded successfully.")

FileNotFoundError: [Errno 2] No such file or directory: 'fraud_detection_model.pkl'

In [27]:
sample_transaction = X_test.iloc[[0]]

prediction = loaded_model.predict(
    sample_transaction
)

probability = loaded_model.predict_proba(
    sample_transaction
)[:, 1]

print("Prediction:", prediction[0])
print("Fraud probability:", probability[0])

NameError: name 'X_test' is not defined

In [28]:
if prediction[0] == 1:
    print("🚨 FRAUDULENT TRANSACTION")
else:
    print("✅ LEGITIMATE TRANSACTION")

NameError: name 'prediction' is not defined